# Template 03: Data Verification

**Purpose:** Verify data quality and generate summary statistics

**Inputs:**
- data/02_conditioned.parquet

**Outputs:**
- results/03_verification_report.txt
- Pass-through (no new checkpoint)

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_comp/v1"


In [3]:
import pandas as pd
import numpy as np
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 03: VERIFICATION")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 03: VERIFICATION
########################################


In [4]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
os.makedirs(f"{output_base}/results", exist_ok=True)

In [5]:
# Validate model config
target_method = cfg.get('model', {}).get('target_method', 'direct')
if target_method == 'residual':
    glm_col = cfg.get('model', {}).get('glm_prediction_column')
    if not glm_col:
        raise ValueError('model.glm_prediction_column required for target_method=residual')
    control_file = cfg.get('data', {}).get('control_model_file')
    if not control_file:
        raise ValueError('data.control_model_file required for target_method=residual')
    print(f'[OK] Config valid: residual method')
    print(f'     GLM column: {glm_col}')
    print(f'     Control file: {control_file}')
else:
    print(f'[OK] Config valid: direct method (no GLM required)')


[OK] Config valid: residual method
     GLM column: pred_pp_comp
     Control file: aux_dataset_car_vindate_ctl_no_duplicate.parquet


In [6]:
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
data = pd.read_parquet(checkpoint_02)
print(f"\n* Loaded: {data.shape}")


* Loaded: (9089028, 112)


In [7]:
# Verification checks
report = []
report.append("="*60)
report.append("DATA VERIFICATION REPORT")
report.append("="*60)
report.append(f"\nShape: {data.shape}")
report.append(f"Rows: {len(data):,}")
report.append(f"Columns: {len(data.columns)}")

# Target stats
target = cfg['experiment']['target']
if target in data.columns:
    report.append(f"\nTarget: {target}")
    report.append(f"  Mean: {data[target].mean():.4f}")
    report.append(f"  Std: {data[target].std():.4f}")
    report.append(f"  Min: {data[target].min():.4f}")
    report.append(f"  Max: {data[target].max():.4f}")

# Fold distribution
fold_col = cfg['data']['fold_column']
if fold_col in data.columns:
    report.append(f"\nFold distribution:")
    for fold, count in data[fold_col].value_counts().sort_index().items():
        report.append(f"  Fold {fold}: {count:,}")

report.append("\n" + "="*60)
report.append("VERIFICATION COMPLETE")
report.append("="*60)

report_text = "\n".join(report)
print(report_text)

DATA VERIFICATION REPORT

Shape: (9089028, 112)
Rows: 9,089,028
Columns: 112

Target: pp_comp
  Mean: 146.8333
  Std: 12799.2492
  Min: 0.0000
  Max: 31022142.0500

Fold distribution:
  Fold 1: 2,279,300
  Fold 2: 2,270,728
  Fold 6: 2,271,182
  Fold 7: 2,267,818

VERIFICATION COMPLETE


In [8]:
# Save report
report_file = f"{output_base}/results/03_verification_report.txt"
with open(report_file, 'w') as f:
    f.write(report_text)

print(f"\nReport saved: {report_file}")


Report saved: output/car_comp/v1/results/03_verification_report.txt


In [9]:
print("\n########################################")
print("# STAGE 03: COMPLETE")
print("########################################")


########################################
# STAGE 03: COMPLETE
########################################
